# 🔨 Crack 3D Reconstruction
Run each cell in order. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Run once per session. Restart runtime if prompted after install.
%pip install -q gradio open3d transformers pillow accelerate opencv-python

In [ ]:
# ── Cell 2: Imports & model load ──────────────────────────────────────────────
import cv2
import numpy as np
import open3d as o3d
import gradio as gr
import torch
from transformers import pipeline
from PIL import Image
import tempfile
import os

# Use GPU if available, otherwise CPU
if torch.cuda.is_available():
    device = 0
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = -1
    print("⚠️  No GPU found — running on CPU (will be slow)")

print("Loading depth model… (first run downloads ~1.3 GB)")
depth_pipe = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=device,
)
print("✅ Model ready.")

In [ ]:
# ── Cell 3: Processing function ───────────────────────────────────────────────

def reconstruct_crack(input_image, crack_intensity=50):
    """Convert a crack photo to a 3D .glb mesh."""
    if input_image is None:
        return None

    try:
        # ── 0. Normalise channels ──────────────────────────────────────────
        # Gradio may deliver RGB or RGBA; Open3D needs uint8 RGB.
        if input_image.ndim == 2:                          # grayscale → RGB
            input_image = np.stack([input_image] * 3, axis=-1)
        elif input_image.shape[2] == 4:                    # RGBA → RGB
            input_image = input_image[:, :, :3]
        input_image = input_image.astype(np.uint8)

        # ── 1. Crack mask → displacement map ──────────────────────────────
        gray = cv2.cvtColor(input_image, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
        mask_blur = cv2.GaussianBlur(mask, (15, 15), 0).astype(np.float32)
        displacement_map = (mask_blur / 255.0) * float(crack_intensity)

        # ── 2. AI depth prediction ─────────────────────────────────────────
        pil = Image.fromarray(input_image)
        depth_raw = depth_pipe(pil)["depth"]               # PIL Image
        base_depth = np.array(depth_raw).astype(np.float32)

        # ── 3. Combine AI depth + crack carving ───────────────────────────
        # Resize displacement to match depth map if sizes differ
        if base_depth.shape != displacement_map.shape:
            displacement_map = cv2.resize(
                displacement_map,
                (base_depth.shape[1], base_depth.shape[0]),
                interpolation=cv2.INTER_LINEAR,
            )
            input_image = cv2.resize(
                input_image,
                (base_depth.shape[1], base_depth.shape[0]),
                interpolation=cv2.INTER_LINEAR,
            )

        final_depth = base_depth + displacement_map
        dmin, dmax = final_depth.min(), final_depth.max()
        depth_norm = (final_depth - dmin) / (dmax - dmin + 1e-6)
        depth_uint = (depth_norm * 1000).astype(np.uint16)

        # ── 4. Build RGBD → point cloud ───────────────────────────────────
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(input_image))
        depth_o3d = o3d.geometry.Image(np.ascontiguousarray(depth_uint))

        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            color_o3d,
            depth_o3d,
            depth_scale=1000.0,
            depth_trunc=3.0,
            convert_rgb_to_intensity=False,
        )

        h, w = depth_uint.shape
        fx = fy = max(h, w)                                # simple pinhole estimate
        intrinsic = o3d.camera.PinholeCameraIntrinsic(
            w, h, fx, fy, w / 2.0, h / 2.0
        )

        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
        pcd.transform([[1, 0, 0, 0],
                       [0, -1, 0, 0],
                       [0, 0, -1, 0],
                       [0, 0,  0, 1]])

        # ── 5. Downsample + denoise ───────────────────────────────────────
        # Voxel downsample keeps point count manageable for Poisson.
        pcd = pcd.voxel_down_sample(voxel_size=0.005)

        # Guard: Poisson needs at least a few thousand points.
        if len(pcd.points) < 500:
            print(f"⚠️  Too few points after downsampling ({len(pcd.points)}). "
                  "Try a higher-resolution image or reduce crack_intensity.")
            return None

        # Statistical outlier removal reduces noise that corrupts Poisson.
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30)
        )
        pcd.orient_normals_towards_camera_location([0, 0, 0])

        # ── 6. Poisson surface reconstruction ────────────────────────────
        # depth=8 is fast and sufficient; depth=9 risks OOM on large clouds.
        mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=8
        )
        bbox = pcd.get_axis_aligned_bounding_box()
        mesh = mesh.crop(bbox)

        # ── 7. Bake vertex colors (vectorized) ───────────────────────────
        # Build a KD-tree once, query all vertices at once via numpy.
        pc_points = np.asarray(pcd.points)
        pc_colors = np.asarray(pcd.colors)
        mesh_verts = np.asarray(mesh.vertices)

        # Batched nearest-neighbor via scipy (much faster than O(n) Open3D loop)
        from scipy.spatial import cKDTree  # available in Colab by default
        tree = cKDTree(pc_points)
        _, idx = tree.query(mesh_verts, workers=-1)        # use all CPU cores
        mesh.vertex_colors = o3d.utility.Vector3dVector(pc_colors[idx])

        # ── 8. Save as .glb (required by Gradio 4+) ──────────────────────
        temp_dir = tempfile.gettempdir()
        out_path = os.path.join(temp_dir, "crack_reconstruction.glb")
        o3d.io.write_triangle_mesh(out_path, mesh)

        print(f"✅ Mesh saved → {out_path} "
              f"({len(mesh.triangles):,} triangles)")
        return out_path

    except Exception as e:
        import traceback
        print("ERROR during reconstruction:")
        traceback.print_exc()
        return None

In [ ]:
# ── Cell 4: Launch Gradio UI ──────────────────────────────────────────────────

with gr.Blocks(title="Crack 3D Reconstruction") as demo:
    gr.Markdown(
        "## 🪨 Crack 3D Reconstruction\n"
        "Upload a photo of a wall crack to generate a textured 3D mesh."
    )

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="numpy", label="Crack Image")
            intensity  = gr.Slider(
                minimum=10, maximum=150, value=50, step=5,
                label="Crack Displacement Strength",
                info="Higher values carve the crack deeper into the mesh.",
            )
            run_btn = gr.Button("Generate 3D Model", variant="primary")

        with gr.Column():
            output_model = gr.Model3D(
                label="3D Crack Mesh",
                clear_color=[0.2, 0.2, 0.2, 1.0],
            )

    run_btn.click(
        fn=reconstruct_crack,
        inputs=[input_img, intensity],
        outputs=output_model,
    )

# share=True creates a public tunnel URL — required to open the UI in Colab.
demo.launch(share=True)